# COCO-pretrained Mask R-CNN — 20 Band Combination Ablation

**Project:** RTS instance segmentation using the 2026 GeoAI Arctic Challenge dataset  
**Model:** Torchvision Mask R-CNN ResNet50-FPN, full `COCO_V1` instance-segmentation weights  
**Input:** 20 selected three-band combinations from the eight available Arctic bands  
**Split:** Existing grouped-by-pixel-overlap split, fold 1, seed 42

This notebook is the Kaggle runner. The reusable Python source is in `src/`, and the experiment list is in `configs/band_combinations.json`. The same folder can be opened in VS Code and committed to GitHub later.

**Important:** Select a Kaggle GPU (T4/P100 or better) and enable Internet before the first run so Torchvision can download the approximately 170 MB COCO checkpoint. Do not upgrade PyTorch/Torchvision in this notebook; Kaggle's preinstalled matched versions should be used.

The first 20-run screen ranks combinations using the selected validation fold only. The fixed local-test set is intentionally not used to choose the winning combination.

In [ ]:
# CELL 1 — Paths and experiment settings
from pathlib import Path
import json
import sys
import os
import gc
import zipfile

INPUT_ROOT = Path("/kaggle/input")

# Discover the attached competition release.
DATA_ROOT = Path(
    "/kaggle/input/datasets/abdullahzaidi1934/"
    "geoai-arctic-challenge-competition-data/competition_release"
)
if not DATA_ROOT.exists():
    metadata_files = list(INPUT_ROOT.rglob("band_names.json"))
    if not metadata_files:
        raise FileNotFoundError("Cannot find metadata/band_names.json. Attach the competition data dataset.")
    DATA_ROOT = metadata_files[0].parent.parent

# Discover this project folder after uploading it as a Kaggle Dataset.
config_files = list(INPUT_ROOT.rglob("band_combinations.json"))
config_files = [p for p in config_files if (p.parent.parent / "src" / "experiment.py").exists()]

# Some Kaggle Dataset uploads expose the ZIP as a file instead of unpacking it.
# In that case, extract this project once into writable /kaggle/working.
if not config_files:
    project_zips = list(INPUT_ROOT.rglob("coco_maskrcnn_band_ablation.zip"))
    if project_zips:
        extracted_root = Path("/kaggle/working/project_source")
        extracted_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(project_zips[0]) as project_archive:
            project_archive.extractall(extracted_root)
        config_files = list(extracted_root.rglob("band_combinations.json"))
        config_files = [p for p in config_files if (p.parent.parent / "src" / "experiment.py").exists()]

if not config_files:
    raise FileNotFoundError(
        "Cannot find this project's configs/band_combinations.json. "
        "Attach the coco_maskrcnn_band_ablation project ZIP as a Kaggle Dataset."
    )
CODE_ROOT = config_files[0].parent.parent

# Reuse the exact grouped split file already used by the RTS project.
split_files = list(INPUT_ROOT.rglob("grouped_folds.json"))
preferred = [p for p in split_files if "rts-mask-r-cnn-band-scale-code" in str(p)]
if preferred:
    SPLIT_PATH = preferred[0]
elif len(split_files) == 1:
    SPLIT_PATH = split_files[0]
else:
    raise FileNotFoundError(
        "Cannot uniquely locate grouped_folds.json. Attach the existing "
        "rts-mask-r-cnn-band-scale-code dataset containing Experiments/splits/grouped_folds.json."
    )

SRC_ROOT = CODE_ROOT / "src"
COMBINATIONS_PATH = CODE_ROOT / "configs" / "band_combinations.json"
OUT_DIR = Path("/kaggle/working/coco_band_ablation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Experiment controls
FOLD = 1                 # same fold used for the earlier RGB baseline (zero-based)
SEED = 42
EPOCHS = 20              # maximum epochs; early stopping may finish earlier
BATCH_SIZE = 2
NUM_WORKERS = 2
USE_AMP = True           # mixed precision on CUDA
PATIENCE = 6
LEARNING_RATE = 0.002
WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9
MIN_SIZE = 256
MAX_SIZE = 512
DETECTIONS_PER_IMAGE = 10
TRAINABLE_BACKBONE_LAYERS = 3  # freezes the ResNet stem + first residual stage
MASK_THRESHOLD = 0.5
MIOU_SCORE_THRESHOLD = 0.5

# None runs all 20. To stage the work, set e.g. ["C01", ..., "C05"].
COMBO_IDS = None
FORCE_RERUN = False

sys.path.insert(0, str(CODE_ROOT))

print("DATA_ROOT :", DATA_ROOT)
print("SPLIT_PATH:", SPLIT_PATH)
print("CODE_ROOT :", CODE_ROOT)
print("OUT_DIR   :", OUT_DIR)
print("Fold / seed:", FOLD, SEED)
print("Combinations:", "ALL 20" if COMBO_IDS is None else COMBO_IDS)


In [ ]:
# CELL 2 — Install only auxiliary packages; keep Kaggle's torch/torchvision pair intact
%pip install -q pycocotools scipy pandas matplotlib

import torch
import torchvision
import numpy as np
import pandas as pd
import matplotlib
import pycocotools
from IPython.display import display

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("A GPU is required for this 20-run Mask R-CNN experiment. Enable GPU in Kaggle Settings.")

from src.data import (
    load_records, load_grouped_split, read_band_indices,
    compute_band_scaling, make_loaders
)
from src.model import build_coco_maskrcnn, count_parameters
from src.experiment import run_experiments
from src.plot_results import plot_all


In [ ]:
# CELL 3 — Verify release, band names, and the saved grouped split
records, coco_data = load_records(DATA_ROOT)
train_records, val_records, local_test_records, split_audit = load_grouped_split(
    records, SPLIT_PATH, FOLD
)
band_indices = read_band_indices(DATA_ROOT)
combinations = json.loads(COMBINATIONS_PATH.read_text())

print("Split method:", split_audit["method"])
print("Fold:", split_audit["fold"])
print("Labelled images:", split_audit["n_labelled"])
print("Training images:", split_audit["n_train"])
print("Validation images:", split_audit["n_val"])
print("Local held-out test images:", split_audit["n_local_test"])
print("Pixel-overlap groups:", split_audit["n_pixel_overlap_groups"])
print("IDs represented in groups:", split_audit["n_ids_in_pixel_overlap_groups"])
print("Groups crossing train/val/local-test:", split_audit["groups_crossing_splits"])
print("Repeated IDs across groups:", split_audit["duplicate_group_ids"])
print("\nBand metadata:", band_indices)
print("\nTwenty experiment combinations:")
display(pd.DataFrame([
    {"ID": c["id"], "Name": c["name"], "Bands": ", ".join(c["bands"])}
    for c in combinations
]))

# These are substantive checks on the connected pixel-overlap groups saved in the split file.
assert split_audit["groups_crossing_splits"] == 0, "A saved pixel-overlap group crosses data partitions."
assert split_audit["duplicate_group_ids"] == 0, "An image ID appears in more than one saved group."
assert split_audit["n_ids_not_listed_in_groups"] == 0, "Some labelled image IDs are missing from the saved pixel-overlap groups."
assert len(combinations) == 20
assert all(len(c["bands"]) == 3 for c in combinations)
assert all(name in band_indices for c in combinations for name in c["bands"])
assert len(train_records) == 523 and len(val_records) == 112 and len(local_test_records) == 121, (
    "The split counts differ from the expected project fold. Stop and inspect the attached split file."
)
print("\nDATA AND SPLIT CHECKS PASSED.")


## COCO model and real-data smoke test

The model below loads the **complete COCO instance-segmentation checkpoint**. We replace only the final COCO class predictors with two-class RTS predictors. The pretrained ResNet stem and first residual stage are frozen; later stages and the detection heads are trainable.

The selected three bands are stretched using training-split-only 2nd/98th percentiles into `[0, 1]`. Torchvision then applies its built-in pretrained mean/std transform. This avoids passing z-scored negative values into a model that expects image tensors in `[0, 1]`.

In [ ]:
# CELL 4 — Compute/cached train-only band scaling and inspect a real batch
SCALING_PATH = OUT_DIR / "preprocessing" / f"fold{FOLD}_seed{SEED}_train_band_scaling.json"
SCALING_PATH.parent.mkdir(parents=True, exist_ok=True)

if SCALING_PATH.exists():
    scaling = json.loads(SCALING_PATH.read_text())
    if scaling.get("n_train_images") != len(train_records):
        raise ValueError("Cached scaling stats do not match the selected training split.")
    print("Loaded cached scaling:", SCALING_PATH)
else:
    print("Computing 2nd/98th percentile scaling from training images only...")
    scaling = compute_band_scaling(train_records, seed=SEED)
    SCALING_PATH.write_text(json.dumps(scaling, indent=2))
    print("Saved:", SCALING_PATH)

c01 = next(c for c in combinations if c["id"] == "C01")
train_loader_smoke, val_loader_smoke, train_ds_smoke, val_ds_smoke = make_loaders(
    train_records, val_records, c01["bands"], band_indices, scaling,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
images_smoke, targets_smoke = next(iter(train_loader_smoke))
print("Batch images:", [tuple(x.shape) for x in images_smoke])
print("Image range:", float(min(x.min() for x in images_smoke)),
      "to", float(max(x.max() for x in images_smoke)))
print("Target boxes:", [tuple(t["boxes"].shape) for t in targets_smoke])
print("Target masks:", [tuple(t["masks"].shape) for t in targets_smoke])
assert all(float(x.min()) >= 0.0 and float(x.max()) <= 1.0 for x in images_smoke)
print("REAL-DATA LOADER CHECK PASSED.")


In [ ]:
# CELL 5 — Load full COCO Mask R-CNN and run one real training-loss/gradient smoke test
import gc

device = torch.device("cuda")
model_smoke = build_coco_maskrcnn(
    detections_per_image=DETECTIONS_PER_IMAGE,
    min_size=MIN_SIZE,
    max_size=MAX_SIZE,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
).to(device)

parameter_counts = count_parameters(model_smoke)
print("Parameters:", {k: round(v / 1e6, 2) for k, v in parameter_counts.items()})
print("Box classes:", model_smoke.roi_heads.box_predictor.cls_score.out_features)
print("Mask classes:", model_smoke.roi_heads.mask_predictor.mask_fcn_logits.out_channels)
print("COCO weights:", "MaskRCNN_ResNet50_FPN_Weights.COCO_V1")
assert model_smoke.roi_heads.box_predictor.cls_score.out_features == 2
assert model_smoke.roi_heads.mask_predictor.mask_fcn_logits.out_channels == 2

model_smoke.train()
smoke_images = [x.to(device) for x in images_smoke]
smoke_targets = [
    {k: v.to(device) if torch.is_tensor(v) else v for k, v in t.items()}
    for t in targets_smoke
]
model_smoke.zero_grad(set_to_none=True)
smoke_losses = model_smoke(smoke_images, smoke_targets)
smoke_total = sum(smoke_losses.values())
smoke_total.backward()
print("Smoke losses:", {k: round(float(v.detach().cpu()), 5) for k, v in smoke_losses.items()})
print("Total loss:", round(float(smoke_total.detach().cpu()), 5))
assert torch.isfinite(smoke_total)
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model_smoke.parameters() if p.requires_grad)
print("COCO MODEL + REAL TRAINING STEP CHECK PASSED.")

del model_smoke, smoke_images, smoke_targets, smoke_losses, smoke_total
# Release smoke-test objects before the 20 actual runs.
del train_loader_smoke, val_loader_smoke, train_ds_smoke, val_ds_smoke
gc.collect()
torch.cuda.empty_cache()
print("Smoke-test memory released.")


## Run the 20 controlled experiments

Each run uses the same grouped fold, seed, training schedule, COCO checkpoint, image scale, anchor setup and evaluation implementation. Only the selected three bands change.

The script saves a `best.pth`, `last_checkpoint.pth` (for epoch-level resuming), training history, validation predictions, sample mask visualisations and one metrics JSON per combination. Completed combinations are skipped on a rerun unless `FORCE_RERUN=True`.

**Do not use the local-test scores to choose among these 20 combinations.** The ranking is based on the validation fold.

In [ ]:
# CELL 6 — Train all 20 combinations (this is the long-running cell)
results_rows = run_experiments(
    data_root=DATA_ROOT,
    split_path=SPLIT_PATH,
    output_dir=OUT_DIR,
    combinations_path=COMBINATIONS_PATH,
    fold=FOLD,
    seed=SEED,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    momentum=MOMENTUM,
    amp=USE_AMP,
    patience=PATIENCE,
    score_threshold=MIOU_SCORE_THRESHOLD,
    mask_threshold=MASK_THRESHOLD,
    min_size=MIN_SIZE,
    max_size=MAX_SIZE,
    detections_per_image=DETECTIONS_PER_IMAGE,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
    force=FORCE_RERUN,
    combo_ids=COMBO_IDS,
)

print("\nCompleted runs in this call:", len(results_rows))
print("Results CSV:", OUT_DIR / "results.csv")


In [ ]:
# CELL 7 — Results table ranked by official validation Mask AP
RESULTS_CSV = OUT_DIR / "results.csv"
if not RESULTS_CSV.exists():
    raise FileNotFoundError("No results.csv yet. Run Cell 6 first.")

results_df = pd.read_csv(RESULTS_CSV).sort_values("AP", ascending=False).reset_index(drop=True)
percentage_columns = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "mIoU"]
for column in percentage_columns:
    if column in results_df:
        results_df[column] = results_df[column] * 100.0

display(results_df[[
    "experiment", "name", "bands", "best_epoch",
    "AP", "AP50", "AP75", "mIoU", "AP_small", "AP_medium", "AP_large"
]])
print("\nCurrent best validation combination:", results_df.iloc[0]["experiment"],
      "—", results_df.iloc[0]["name"])


In [ ]:
# CELL 8 — Generate ranked graphs (highest bar extends furthest to the right)
from IPython.display import display, Image

FIGURE_DIR = OUT_DIR / "figures"
saved_figures = plot_all(RESULTS_CSV, FIGURE_DIR)
print("Saved graph files:")
for path in saved_figures:
    print(path)

for name in ["metrics_dashboard.png", "ranked_AP.png", "ranked_AP50.png", "ranked_AP75.png", "ranked_mIoU.png", "ranked_AP_by_size.png"]:
    image_path = FIGURE_DIR / name
    if image_path.exists():
        print(name)
        display(Image(filename=str(image_path)))


In [ ]:
# CELL 9 — Inspect the validation prediction visualisations for the top combinations
from IPython.display import display, Image

ranked_for_gallery = pd.read_csv(RESULTS_CSV).sort_values("AP", ascending=False)
for _, result in ranked_for_gallery.head(5).iterrows():
    exp_id = result["experiment"]
    sample_path = OUT_DIR / "runs" / exp_id / "samples_val.png"
    if sample_path.exists():
        print(f"{exp_id} | {result['name']} | AP={result['AP']*100:.2f}%")
        display(Image(filename=str(sample_path)))


In [ ]:
# CELL 10 — Create a downloadable results archive (checkpoints intentionally excluded)
import zipfile

ARCHIVE_PATH = Path("/kaggle/working/coco_band_ablation_results.zip")
include_roots = [OUT_DIR / "results.csv", OUT_DIR / "experiment_config.json", OUT_DIR / "preprocessing"]
with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in include_roots:
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.relative_to(OUT_DIR))
        elif file_path.is_dir():
            for file in file_path.rglob("*"):
                if file.is_file():
                    archive.write(file, arcname=file.relative_to(OUT_DIR))
    for folder in [OUT_DIR / "runs", FIGURE_DIR]:
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file() and file.suffix.lower() not in {".pth", ".pt"} and "last_checkpoint" not in file.name:
                    archive.write(file, arcname=file.relative_to(OUT_DIR))

print("Results archive:", ARCHIVE_PATH)
print("Archive size (MB):", round(ARCHIVE_PATH.stat().st_size / (1024**2), 2))
from IPython.display import FileLink, display
display(FileLink(str(ARCHIVE_PATH)))


## What to preserve

- Keep the entire `/kaggle/working/coco_band_ablation/` folder as a Kaggle notebook output/version so the checkpoints and histories are preserved.
- Download `coco_band_ablation_results.zip` for tables, plots, per-combination validation predictions and sample visualisations. It excludes the large `.pth` checkpoints.
- The best checkpoints are in `coco_band_ablation/runs/C01/best.pth` through `C20/best.pth`.
- Do not push model checkpoints or large generated outputs to GitHub. Push the notebook, `src/`, `configs/`, README and small result summaries instead.